# WiDS 2026 | Edge-Aware Survival Superblend


In [1]:
import os
import glob
import math
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.neighbors import NearestNeighbors

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
import xgboost as xgb


RUN_MODE = "full"   # "fast" or "full"
N_SPLITS = 5 if RUN_MODE == "full" else 4
SEEDS = (42, 2026) if RUN_MODE == "full" else (42,)
OUTPUT_PATH = "/kaggle/working/submission.csv" if os.path.isdir("/kaggle/working") else "submission.csv"

LEGACY_BLEND_12 = 0.05
LEGACY_BLEND_24 = 0.08
LEGACY_BLEND_48 = 0.08

PAIRWISE_REG = 0.02
PAIRWISE_BRIER12 = 0.05

SOFT_GATE_CENTERS = {24: 5000.0, 48: 6000.0}
SOFT_GATE_WIDTHS = {24: 1400.0, 48: 1900.0}


def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    for root, _, files in os.walk("/kaggle/input"):
        files = set(files)
        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
            return root
    raise FileNotFoundError("Could not find competition data directory.")

DATA_DIR = find_data_dir()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))


def sigmoid(x):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-x))

def soft_gate(x, center, width):
    width = max(float(width), 1e-6)
    return sigmoid((center - np.asarray(x, dtype=float)) / width)

def enforce_monotonicity(arr):
    arr = np.clip(np.asarray(arr, dtype=float), 0.0, 1.0)
    return np.maximum.accumulate(arr, axis=1)

def make_binary_target(time_vals, event_vals, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_brier(time_vals, event_vals, prob, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    prob = np.clip(np.asarray(prob, dtype=float), 0.0, 1.0)
    valid = ~((event_vals == 0) & (time_vals < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event_vals == 1) & (time_vals <= horizon)).astype(float)[valid]
    return float(np.mean((prob[valid] - y_true) ** 2))

def compute_c_index(time_vals, event_vals, risk):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    risk = np.asarray(risk, dtype=float)
    n = len(time_vals)
    conc = 0.0
    comp = 0.0
    for i in range(n):
        if event_vals[i] != 1:
            continue
        for j in range(n):
            if i == j or time_vals[i] >= time_vals[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_hybrid(prob12, prob24, prob48, prob72):
    time_vals = train_df["time_to_hit_hours"].values
    event_vals = train_df["event"].values
    c_idx = compute_c_index(time_vals, event_vals, prob12)
    b24 = compute_brier(time_vals, event_vals, prob24, 24)
    b48 = compute_brier(time_vals, event_vals, prob48, 48)
    b72 = compute_brier(time_vals, event_vals, prob72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    score = 0.3 * c_idx + 0.7 * (1.0 - wb)
    return score, c_idx, wb, b24, b48, b72

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0

    w = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            w[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            w[i] = 1.0 / G(horizon)
        else:
            w[i] = 0.0
    return w

def fit_with_optional_weight(model, X, y, sw=None):
    try:
        if sw is None:
            model.fit(X, y)
        else:
            model.fit(X, y, sample_weight=sw)
        return model
    except Exception:
        if isinstance(model, Pipeline):
            step_name = model.steps[-1][0]
            kwargs = {}
            if sw is not None:
                kwargs[f"{step_name}__sample_weight"] = sw
            model.fit(X, y, **kwargs)
            return model
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if isinstance(p, list):
            p = np.asarray(p)
        if getattr(p, "ndim", 1) == 2:
            return np.asarray(p[:, 1], dtype=float)
        return np.asarray(p, dtype=float).reshape(-1)
    return np.asarray(model.predict(X), dtype=float).reshape(-1)

def pct_rank(values, ref_values):
    ref_values = np.asarray(ref_values, dtype=float)
    if ref_values.size == 0:
        return np.zeros(len(values), dtype=float)
    ref_sorted = np.sort(ref_values)
    return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / max(1, ref_sorted.size)


def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = r["closing_speed_m_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0.0)
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"].clip(lower=0.0)
    area_growth = r["area_growth_rate_ha_per_h"]
    align = r["alignment_abs"]
    fire_radius_m = np.sqrt(area_first * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0.0)
    eff_close = closing_pos + radial

    r["fire_radius_m"] = fire_radius_m
    r["fire_radius_km"] = fire_radius_m / 1000.0

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["sqrt_distance"] = np.sqrt(dist)
    r["inv_distance"] = 1.0 / (r["dist_km"] + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["edge_dist_m"] = dist - fire_radius_m
    r["margin_5km_m"] = r["edge_dist_m"] - 5000.0
    r["margin_ratio"] = r["margin_5km_m"] / (dist + 1.0)
    r["inside_edge_5km"] = (r["edge_dist_m"] <= 5000.0).astype(float)
    r["inside_edge_3km"] = (r["edge_dist_m"] <= 3000.0).astype(float)
    r["inside_edge_1km"] = (r["edge_dist_m"] <= 1000.0).astype(float)

    r["area_to_dist_ratio"] = area_first / (r["dist_km"] + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)
    r["radius_to_dist"] = fire_radius_m / dist

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])
    r["edge_eta"] = np.where(eff_close > 0.1, np.maximum(r["margin_5km_m"], 0.0) / (eff_close + 1e-3), 9999.0).clip(max=9999.0)
    r["log_edge_eta"] = np.log1p(r["edge_eta"])

    r["threat_score"] = align * speed / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * speed
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = speed * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0.0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0.0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0.0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0.0)

    for H in [12, 24, 48, 72]:
        r[f"edge_gap_h{H}"] = r["margin_5km_m"] - eff_close * H
        r[f"edge_sig_h{H}"] = sigmoid(-r[f"edge_gap_h{H}"] / 1500.0)

    r["zone_3km"] = (dist < 3000.0).astype(float)
    r["zone_near"] = (dist < 5000.0).astype(float)
    r["zone_8km"] = (dist < 8000.0).astype(float)
    r["zone_warning"] = ((dist >= 5000.0) & (dist < 10000.0)).astype(float)
    r["zone_far"] = (dist >= 10000.0).astype(float)
    r["zone_20km"] = (dist < 20000.0).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1.0).values
    ref_eff = (
        ref["closing_speed_m_per_h"].clip(lower=0.0).values
        + ref["radial_growth_rate_m_per_h"].clip(lower=0.0).values
    )
    ref_margin = (
        ref["dist_min_ci_0_5h"].clip(lower=1.0).values
        - np.sqrt(np.maximum(ref["area_first_ha"].values, 0.0) * 10000.0 / np.pi)
        - 5000.0
    )
    ref_threat = ref["alignment_abs"].values * ref_eff / np.log1p(ref_dist)

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["margin_rank_global"] = pct_rank(r["margin_5km_m"].values, ref_margin)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref_dist < 10000.0
    r["near_speed_rank"] = pct_rank(eff_close.values, ref_eff[ref_near]) if ref_near.sum() else 0.0
    r["near_margin_rank"] = pct_rank(r["margin_5km_m"].values, ref_margin[ref_near]) if ref_near.sum() else 0.0

    ref_far = ref_dist >= 10000.0
    r["far_threat_rank"] = pct_rank(r["threat_score_eff"].values, ref_threat[ref_far]) if ref_far.sum() else 0.0

    return r

train_proc = create_features(train_df, train_df)
test_proc = create_features(test_df, train_df)

DROP_COLS = ["event_id", "event", "time_to_hit_hours"]
ALL_FEATURES = [c for c in train_proc.columns if c not in DROP_COLS]

LINEAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "fire_radius_km", "edge_dist_m", "margin_5km_m", "margin_ratio",
    "effective_closing_speed", "eta_effective", "log_eta_effective",
    "alignment_abs", "threat_score_eff",
    "area_to_dist_ratio", "radius_to_dist",
    "num_perimeters_0_5h", "dt_first_last_0_5h",
    "low_temporal_resolution_0_5h",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "inside_edge_5km", "inside_edge_3km", "zone_near", "zone_warning",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

TREE_FEATURES = sorted(set(ALL_FEATURES) - {"spread_bearing_deg"})

TIME_MODEL_FEATURES = [
    "dist_km", "fire_radius_km", "edge_dist_m", "margin_5km_m",
    "edge_eta", "log_edge_eta", "eta_effective", "log_eta_effective",
    "effective_closing_speed", "alignment_abs",
    "area_first_ha", "area_growth_rate_ha_per_h",
    "radial_growth_rate_m_per_h", "closing_speed_m_per_h",
    "num_perimeters_0_5h", "dt_first_last_0_5h",
    "projected_advance_pos", "slope_toward", "accel_toward",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "near_margin_rank", "near_speed_rank", "dist_rank_global",
]
TIME_MODEL_FEATURES = [c for c in TIME_MODEL_FEATURES if c in train_proc.columns]

X_tree_train = train_proc[TREE_FEATURES].copy()
X_tree_test = test_proc[TREE_FEATURES].copy()

X_lin_train = train_proc[LINEAR_FEATURES].copy()
X_lin_test = test_proc[LINEAR_FEATURES].copy()

X_time_train = train_proc[TIME_MODEL_FEATURES].copy()
X_time_test = test_proc[TIME_MODEL_FEATURES].copy()

time_values = train_df["time_to_hit_hours"].values.astype(float)
event_values = train_df["event"].values.astype(int)
dist_train = train_df["dist_min_ci_0_5h"].values.astype(float)
dist_test = test_df["dist_min_ci_0_5h"].values.astype(float)


def build_logit(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=0.8, penalty="l2", solver="lbfgs", max_iter=5000)),
    ])

def build_lgb(h, seed):
    params = {
        12: dict(
            objective="binary",
            n_estimators=320 if RUN_MODE == "full" else 180,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=6,
            min_child_samples=8,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.6,
            reg_lambda=2.0,
        ),
        24: dict(
            objective="binary",
            n_estimators=380 if RUN_MODE == "full" else 220,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=6,
            min_child_samples=10,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.8,
            reg_lambda=2.5,
        ),
        48: dict(
            objective="binary",
            n_estimators=420 if RUN_MODE == "full" else 260,
            learning_rate=0.025,
            max_depth=4,
            num_leaves=8,
            min_child_samples=12,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=1.0,
            reg_lambda=3.0,
        ),
        72: dict(
            objective="binary",
            n_estimators=280 if RUN_MODE == "full" else 160,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=6,
            min_child_samples=10,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.8,
            reg_lambda=2.5,
        ),
    }[h]
    params.update(dict(random_state=seed, n_jobs=-1, verbose=-1))
    return lgb.LGBMClassifier(**params)

def build_cat(h, seed):
    params = {
        12: dict(iterations=420 if RUN_MODE == "full" else 240, depth=4, learning_rate=0.03, l2_leaf_reg=8.0),
        24: dict(iterations=460 if RUN_MODE == "full" else 260, depth=4, learning_rate=0.03, l2_leaf_reg=10.0),
        48: dict(iterations=520 if RUN_MODE == "full" else 300, depth=5, learning_rate=0.025, l2_leaf_reg=12.0),
        72: dict(iterations=380 if RUN_MODE == "full" else 220, depth=4, learning_rate=0.03, l2_leaf_reg=10.0),
    }[h]
    params.update(dict(
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    ))
    return CatBoostClassifier(**params)

def build_xgb(h, seed):
    params = {
        12: dict(n_estimators=280 if RUN_MODE == "full" else 160, max_depth=3, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.6, reg_lambda=2.0, min_child_weight=5),
        24: dict(n_estimators=320 if RUN_MODE == "full" else 180, max_depth=3, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.8, reg_lambda=2.5, min_child_weight=6),
        48: dict(n_estimators=360 if RUN_MODE == "full" else 220, max_depth=4, learning_rate=0.025, subsample=0.90, colsample_bytree=0.90, reg_alpha=1.0, reg_lambda=3.0, min_child_weight=8),
        72: dict(n_estimators=260 if RUN_MODE == "full" else 150, max_depth=3, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.8, reg_lambda=2.5, min_child_weight=6),
    }[h]
    params.update(dict(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=seed,
        verbosity=0,
    ))
    return xgb.XGBClassifier(**params)

def build_time_regressor(seed):
    return CatBoostRegressor(
        iterations=520 if RUN_MODE == "full" else 280,
        depth=5,
        learning_rate=0.03,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )

def repeated_isotonic(train_signal, test_signal, y, mask, seeds):
    n_train = len(train_signal)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    valid_idx = np.where(mask)[0]
    xv = np.asarray(train_signal, dtype=float)[valid_idx]
    yv = np.asarray(y, dtype=int)[valid_idx]

    for seed in seeds:
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(np.asarray(train_signal, dtype=float)[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(np.asarray(train_signal, dtype=float)[va_idx])
            cnt[va_idx] += 1.0
        ir_full = IsotonicRegression(out_of_bounds="clip")
        ir_full.fit(xv, yv)
        test_pred += ir_full.predict(np.asarray(test_signal, dtype=float))

    nz = cnt > 0
    oof[nz] /= cnt[nz]
    test_pred /= max(1, len(seeds))

    if (~nz).any():
        ir_full = IsotonicRegression(out_of_bounds="clip")
        ir_full.fit(xv, yv)
        oof[~nz] = ir_full.predict(np.asarray(train_signal, dtype=float)[~nz])

    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def repeated_binary_model(X_train, X_test, y, mask, model_builder, seeds, horizon, use_ipcw=False):
    n_train = len(X_train)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = np.asarray(y, dtype=int)[valid_idx]

    for seed in seeds:
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = model_builder(seed)
            sw = None
            if use_ipcw:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = model_builder(seed)
        sw_full = None
        if use_ipcw:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        test_pred += safe_predict_proba(model_full, X_test)

    nz = cnt > 0
    oof[nz] /= cnt[nz]
    test_pred /= max(1, len(seeds))

    if (~nz).any():
        model_full = model_builder(99991)
        sw_full = None
        if use_ipcw:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        oof[~nz] = safe_predict_proba(model_full, X_train.iloc[~nz])

    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


gate_signal_train = (
    0.45 * train_proc["edge_sig_h72"].values
    + 0.35 * sigmoid(-train_proc["margin_5km_m"].values / 3000.0)
    + 0.20 * sigmoid(-(train_proc["edge_gap_h48"].values) / 2500.0)
)
gate_signal_test = (
    0.45 * test_proc["edge_sig_h72"].values
    + 0.35 * sigmoid(-test_proc["margin_5km_m"].values / 3000.0)
    + 0.20 * sigmoid(-(test_proc["edge_gap_h48"].values) / 2500.0)
)

gate_logit_oof, gate_logit_test = repeated_binary_model(
    X_lin_train, X_lin_test,
    train_df["event"].values.astype(int),
    np.ones(len(train_df), dtype=bool),
    lambda seed: build_logit(seed),
    SEEDS,
    horizon=72,
    use_ipcw=False,
)
gate_iso_oof, gate_iso_test = repeated_isotonic(
    gate_signal_train, gate_signal_test,
    train_df["event"].values.astype(int),
    np.ones(len(train_df), dtype=bool),
    SEEDS,
)
gate72_oof = np.clip(0.55 * gate_logit_oof + 0.45 * gate_iso_oof, 0.0, 1.0)
gate72_test = np.clip(0.55 * gate_logit_test + 0.45 * gate_iso_test, 0.0, 1.0)

signal_store_oof = {}
signal_store_test = {}

for H in [12, 24, 48]:
    y_h, mask_h = make_binary_target(time_values, event_values, H)

    physics_raw_train = (
        0.50 * train_proc[f"edge_sig_h{H}"].values
        + 0.25 * sigmoid(-(train_proc["margin_5km_m"].values - train_proc["effective_closing_speed"].values * H) / 2200.0)
        + 0.25 * sigmoid(-(train_proc["edge_eta"].values - H) / (0.30 * H + 4.0))
    )
    physics_raw_test = (
        0.50 * test_proc[f"edge_sig_h{H}"].values
        + 0.25 * sigmoid(-(test_proc["margin_5km_m"].values - test_proc["effective_closing_speed"].values * H) / 2200.0)
        + 0.25 * sigmoid(-(test_proc["edge_eta"].values - H) / (0.30 * H + 4.0))
    )

    timing_raw_train = (
        0.60 * sigmoid(-(train_proc["edge_eta"].values - H) / (0.22 * H + 3.0))
        + 0.40 * sigmoid(-(train_proc["eta_effective"].values - H) / (0.35 * H + 5.0))
    )
    timing_raw_test = (
        0.60 * sigmoid(-(test_proc["edge_eta"].values - H) / (0.22 * H + 3.0))
        + 0.40 * sigmoid(-(test_proc["eta_effective"].values - H) / (0.35 * H + 5.0))
    )

    physics_oof, physics_test = repeated_isotonic(physics_raw_train, physics_raw_test, y_h, mask_h, SEEDS)
    timing_oof, timing_test = repeated_isotonic(timing_raw_train, timing_raw_test, y_h, mask_h, SEEDS)

    signal_store_oof[f"physics_{H}"] = physics_oof
    signal_store_test[f"physics_{H}"] = physics_test
    signal_store_oof[f"timing_{H}"] = timing_oof
    signal_store_test[f"timing_{H}"] = timing_test

binary_oof = {}
binary_test = {}

for H in [12, 24, 48]:
    y_h, mask_h = make_binary_target(time_values, event_values, H)

    lgb_oof, lgb_test = repeated_binary_model(
        X_tree_train, X_tree_test, y_h, mask_h,
        lambda seed, h=H: build_lgb(h, seed),
        SEEDS, horizon=H, use_ipcw=(H in [24, 48]),
    )
    cat_oof, cat_test = repeated_binary_model(
        X_tree_train, X_tree_test, y_h, mask_h,
        lambda seed, h=H: build_cat(h, seed),
        SEEDS, horizon=H, use_ipcw=(H in [24, 48]),
    )
    xgb_oof, xgb_test = repeated_binary_model(
        X_tree_train, X_tree_test, y_h, mask_h,
        lambda seed, h=H: build_xgb(h, seed),
        SEEDS, horizon=H, use_ipcw=(H in [24, 48]),
    )
    lin_oof, lin_test = repeated_binary_model(
        X_lin_train, X_lin_test, y_h, mask_h,
        lambda seed: build_logit(seed),
        SEEDS, horizon=H, use_ipcw=(H in [24, 48]),
    )

    binary_oof[f"lgb_{H}"] = lgb_oof
    binary_test[f"lgb_{H}"] = lgb_test
    binary_oof[f"cat_{H}"] = cat_oof
    binary_test[f"cat_{H}"] = cat_test
    binary_oof[f"xgb_{H}"] = xgb_oof
    binary_test[f"xgb_{H}"] = xgb_test
    binary_oof[f"lin_{H}"] = lin_oof
    binary_test[f"lin_{H}"] = lin_test

def event_time_knn_oof_test(H, gate_oof, gate_test):
    oof = np.zeros(len(train_df), dtype=float)
    test_pred = np.zeros(len(test_df), dtype=float)
    cnt = np.zeros(len(train_df), dtype=float)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=777)
    dummy = event_values

    for tr_idx, va_idx in skf.split(X_time_train, dummy):
        pos_tr = tr_idx[event_values[tr_idx] == 1]
        if len(pos_tr) < 5:
            continue

        scaler = StandardScaler()
        X_pos = scaler.fit_transform(X_time_train.iloc[pos_tr])
        neigh = NearestNeighbors(n_neighbors=min(10 if RUN_MODE == "full" else 7, len(pos_tr)), metric="euclidean")
        neigh.fit(X_pos)
        t_pos = time_values[pos_tr]

        X_va = scaler.transform(X_time_train.iloc[va_idx])
        d_va, i_va = neigh.kneighbors(X_va)
        d_va = np.maximum(d_va, 1e-6)
        w_va = 1.0 / d_va
        w_va /= w_va.sum(axis=1, keepdims=True)
        pred_va = (w_va * (t_pos[i_va] <= H)).sum(axis=1)
        pred_va *= gate_oof[va_idx]

        oof[va_idx] += pred_va
        cnt[va_idx] += 1.0

        X_te = scaler.transform(X_time_test)
        d_te, i_te = neigh.kneighbors(X_te)
        d_te = np.maximum(d_te, 1e-6)
        w_te = 1.0 / d_te
        w_te /= w_te.sum(axis=1, keepdims=True)
        pred_te = (w_te * (t_pos[i_te] <= H)).sum(axis=1)
        pred_te *= gate_test
        test_pred += pred_te / N_SPLITS

    nz = cnt > 0
    oof[nz] /= cnt[nz]

    if (~nz).any():
        pos_idx = np.where(event_values == 1)[0]
        scaler = StandardScaler()
        X_pos = scaler.fit_transform(X_time_train.iloc[pos_idx])
        neigh = NearestNeighbors(n_neighbors=min(12 if RUN_MODE == "full" else 8, len(pos_idx)), metric="euclidean")
        neigh.fit(X_pos)
        t_pos = time_values[pos_idx]

        X_fill = scaler.transform(X_time_train.iloc[~nz])
        d_fill, i_fill = neigh.kneighbors(X_fill)
        d_fill = np.maximum(d_fill, 1e-6)
        w_fill = 1.0 / d_fill
        w_fill /= w_fill.sum(axis=1, keepdims=True)
        oof[~nz] = (w_fill * (t_pos[i_fill] <= H)).sum(axis=1) * gate_oof[~nz]

    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def event_time_regressor_oof_test(H, gate_oof, gate_test):
    oof_mu = np.zeros(len(train_df), dtype=float)
    cnt = np.zeros(len(train_df), dtype=float)
    test_mu = np.zeros(len(test_df), dtype=float)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=909)
    dummy = event_values

    for fold_id, (tr_idx, va_idx) in enumerate(skf.split(X_time_train, dummy)):
        pos_tr = tr_idx[event_values[tr_idx] == 1]
        if len(pos_tr) < 8:
            continue

        model = build_time_regressor(1000 + fold_id)
        y_tr = np.log1p(time_values[pos_tr])
        fit_with_optional_weight(model, X_time_train.iloc[pos_tr], y_tr, None)

        mu_va = np.asarray(model.predict(X_time_train.iloc[va_idx]), dtype=float)
        mu_te = np.asarray(model.predict(X_time_test), dtype=float)

        oof_mu[va_idx] += mu_va
        cnt[va_idx] += 1.0
        test_mu += mu_te / N_SPLITS

    nz = cnt > 0
    oof_mu[nz] /= cnt[nz]

    if (~nz).any():
        pos_idx = np.where(event_values == 1)[0]
        model = build_time_regressor(20260)
        y_full = np.log1p(time_values[pos_idx])
        fit_with_optional_weight(model, X_time_train.iloc[pos_idx], y_full, None)
        oof_mu[~nz] = np.asarray(model.predict(X_time_train.iloc[~nz]), dtype=float)

    pos_mask = event_values == 1
    resid = np.log1p(time_values[pos_mask]) - oof_mu[pos_mask]
    sigma = float(np.nanstd(resid))
    sigma = max(min(sigma, 1.25), 0.30)

    prob_oof = gate_oof * norm.cdf((math.log1p(H) - oof_mu) / sigma)
    prob_test = gate_test * norm.cdf((math.log1p(H) - test_mu) / sigma)

    return np.clip(prob_oof, 0.0, 1.0), np.clip(prob_test, 0.0, 1.0)

time_model_oof = {}
time_model_test = {}

for H in [12, 24, 48]:
    knn_oof, knn_test = event_time_knn_oof_test(H, gate72_oof, gate72_test)
    time_oof, time_test = event_time_regressor_oof_test(H, gate72_oof, gate72_test)

    time_model_oof[f"knn_{H}"] = knn_oof
    time_model_test[f"knn_{H}"] = knn_test
    time_model_oof[f"timecat_{H}"] = time_oof
    time_model_test[f"timecat_{H}"] = time_test

def prior_to_vector(prior_dict, names):
    w = np.array([prior_dict.get(n, 0.0) for n in names], dtype=float)
    w = np.clip(w, 0.0, None)
    if w.sum() <= 0:
        w = np.ones(len(names), dtype=float)
    return w / w.sum()

def optimize_weights_brier(P, y, prior, reg=0.02, shrink=0.35):
    P = np.clip(np.asarray(P, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)

    def objective(w):
        pred = P @ w
        return float(np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2))

    if P.shape[1] == 1:
        return prior.copy(), prior.copy()

    bounds = [(0.0, 1.0)] * P.shape[1]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    try:
        res = minimize(
            objective,
            prior.copy(),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 400, "ftol": 1e-10},
        )
        opt = np.asarray(res.x if res.success else prior, dtype=float)
    except Exception:
        opt = prior.copy()

    opt = np.clip(opt, 0.0, None)
    opt = opt / max(opt.sum(), 1e-9)
    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 0.0, None)
    final = final / max(final.sum(), 1e-9)
    return final, opt

def make_comparable_pairs(time_vals, event_vals):
    left, right = [], []
    n = len(time_vals)
    for i in range(n):
        if event_vals[i] != 1:
            continue
        for j in range(n):
            if i == j or time_vals[i] >= time_vals[j]:
                continue
            left.append(i)
            right.append(j)
    return np.asarray(left, dtype=int), np.asarray(right, dtype=int)

PAIR_L, PAIR_R = make_comparable_pairs(time_values, event_values)

def optimize_weights_pairwise(P, prior, extra_brier_target=None, shrink=0.30):
    P = np.asarray(P, dtype=float)
    if P.shape[1] == 1:
        return prior.copy(), prior.copy()

    diff = P[PAIR_L] - P[PAIR_R]
    y12 = None if extra_brier_target is None else np.asarray(extra_brier_target, dtype=float)

    def objective(w):
        margin = diff @ w
        pair_loss = np.mean(np.log1p(np.exp(-np.clip(margin, -50.0, 50.0))))
        reg_loss = PAIRWISE_REG * np.sum((w - prior) ** 2)
        if y12 is not None:
            pred = P @ w
            reg_loss += PAIRWISE_BRIER12 * np.mean((pred - y12) ** 2)
        return float(pair_loss + reg_loss)

    bounds = [(0.0, 1.0)] * P.shape[1]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    try:
        res = minimize(
            objective,
            prior.copy(),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 600, "ftol": 1e-10},
        )
        opt = np.asarray(res.x if res.success else prior, dtype=float)
    except Exception:
        opt = prior.copy()

    opt = np.clip(opt, 0.0, None)
    opt = opt / max(opt.sum(), 1e-9)
    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 0.0, None)
    final = final / max(final.sum(), 1e-9)
    return final, opt

weights_log = {}

y12, _ = make_binary_target(time_values, event_values, 12)
names12 = [
    "physics_12", "timing_12",
    "lgb_12", "cat_12", "xgb_12", "lin_12",
    "knn_12", "timecat_12",
]
P12_oof = np.column_stack([
    signal_store_oof["physics_12"],
    signal_store_oof["timing_12"],
    binary_oof["lgb_12"], binary_oof["cat_12"], binary_oof["xgb_12"], binary_oof["lin_12"],
    time_model_oof["knn_12"], time_model_oof["timecat_12"],
])
P12_test = np.column_stack([
    signal_store_test["physics_12"],
    signal_store_test["timing_12"],
    binary_test["lgb_12"], binary_test["cat_12"], binary_test["xgb_12"], binary_test["lin_12"],
    time_model_test["knn_12"], time_model_test["timecat_12"],
])
prior12 = prior_to_vector({
    "physics_12": 0.12, "timing_12": 0.13,
    "lgb_12": 0.14, "cat_12": 0.10, "xgb_12": 0.10, "lin_12": 0.07,
    "knn_12": 0.17, "timecat_12": 0.17,
}, names12)
w12, opt12 = optimize_weights_pairwise(P12_oof, prior12, extra_brier_target=y12, shrink=0.25)
prob12_oof = np.clip(P12_oof @ w12, 0.0, 1.0)
prob12_test = np.clip(P12_test @ w12, 0.0, 1.0)
weights_log["12h"] = dict(names=names12, prior=prior12.tolist(), opt=opt12.tolist(), final=w12.tolist())

for H in [24, 48]:
    y_h, mask_h = make_binary_target(time_values, event_values, H)

    names = [
        f"physics_{H}", f"timing_{H}",
        f"lgb_{H}", f"cat_{H}", f"xgb_{H}", f"lin_{H}",
        f"knn_{H}", f"timecat_{H}",
    ]
    P_oof = np.column_stack([
        signal_store_oof[f"physics_{H}"],
        signal_store_oof[f"timing_{H}"],
        binary_oof[f"lgb_{H}"], binary_oof[f"cat_{H}"], binary_oof[f"xgb_{H}"], binary_oof[f"lin_{H}"],
        time_model_oof[f"knn_{H}"], time_model_oof[f"timecat_{H}"],
    ])
    P_test = np.column_stack([
        signal_store_test[f"physics_{H}"],
        signal_store_test[f"timing_{H}"],
        binary_test[f"lgb_{H}"], binary_test[f"cat_{H}"], binary_test[f"xgb_{H}"], binary_test[f"lin_{H}"],
        time_model_test[f"knn_{H}"], time_model_test[f"timecat_{H}"],
    ])

    gate_train = soft_gate(dist_train, SOFT_GATE_CENTERS[H], SOFT_GATE_WIDTHS[H])
    gate_test = soft_gate(dist_test, SOFT_GATE_CENTERS[H], SOFT_GATE_WIDTHS[H])

    near_mask = mask_h & ((dist_train < 8000.0) | (event_values == 1))
    far_mask = mask_h & (dist_train >= 5000.0)

    prior_near = prior_to_vector({
        f"physics_{H}": 0.10,
        f"timing_{H}": 0.13,
        f"lgb_{H}": 0.16,
        f"cat_{H}": 0.11,
        f"xgb_{H}": 0.11,
        f"lin_{H}": 0.07,
        f"knn_{H}": 0.15,
        f"timecat_{H}": 0.17,
    }, names)
    prior_far = prior_to_vector({
        f"physics_{H}": 0.20,
        f"timing_{H}": 0.12,
        f"lgb_{H}": 0.19,
        f"cat_{H}": 0.14,
        f"xgb_{H}": 0.14,
        f"lin_{H}": 0.12,
        f"knn_{H}": 0.04,
        f"timecat_{H}": 0.05,
    }, names)

    w_near, opt_near = optimize_weights_brier(P_oof[near_mask], y_h[near_mask], prior_near, reg=0.04 if H == 24 else 0.03, shrink=0.30 if H == 24 else 0.35)
    w_far, opt_far = optimize_weights_brier(P_oof[far_mask], y_h[far_mask], prior_far, reg=0.04 if H == 24 else 0.03, shrink=0.45 if H == 24 else 0.50)

    pred_near_oof = np.clip(P_oof @ w_near, 0.0, 1.0)
    pred_near_test = np.clip(P_test @ w_near, 0.0, 1.0)
    pred_far_oof = np.clip(P_oof @ w_far, 0.0, 1.0)
    pred_far_test = np.clip(P_test @ w_far, 0.0, 1.0)

    if H == 24:
        prob24_oof = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
        prob24_test = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test
    else:
        prob48_oof = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
        prob48_test = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    weights_log[f"{H}h_near"] = dict(names=names, prior=prior_near.tolist(), opt=opt_near.tolist(), final=w_near.tolist())
    weights_log[f"{H}h_far"] = dict(names=names, prior=prior_far.tolist(), opt=opt_far.tolist(), final=w_far.tolist())

def find_legacy_prediction_files():
    roots = [DATA_DIR, "/kaggle/input", "."]
    found = []
    seen = set()
    for root in roots:
        if not os.path.exists(root):
            continue
        pattern = os.path.join(root, "**", "samplecsv_sub*.csv")
        for path in glob.glob(pattern, recursive=True):
            if path not in seen:
                found.append(path)
                seen.add(path)
    return sorted(found)

legacy_paths = find_legacy_prediction_files()
legacy_frames = []
for path in legacy_paths:
    try:
        df_legacy = pd.read_csv(path)
        cols = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
        if all(c in df_legacy.columns for c in cols) and len(df_legacy) == len(test_df):
            df_legacy = df_legacy[cols].copy()
            df_legacy = df_legacy.merge(test_df[["event_id"]], on="event_id", how="right")
            if df_legacy["event_id"].is_unique and not df_legacy.isna().any().any():
                legacy_frames.append(df_legacy.sort_values("event_id").reset_index(drop=True))
    except Exception:
        pass

pred_test_sorted = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": prob12_test,
    "prob_24h": prob24_test,
    "prob_48h": prob48_test,
}).sort_values("event_id").reset_index(drop=True)

if legacy_frames:
    legacy_stack = np.stack([
        f.sort_values("event_id")[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
        for f in legacy_frames
    ], axis=0)
    legacy_mean = legacy_stack.mean(axis=0)
    pred_test_sorted["prob_12h"] = (1.0 - LEGACY_BLEND_12) * pred_test_sorted["prob_12h"].values + LEGACY_BLEND_12 * legacy_mean[:, 0]
    pred_test_sorted["prob_24h"] = (1.0 - LEGACY_BLEND_24) * pred_test_sorted["prob_24h"].values + LEGACY_BLEND_24 * legacy_mean[:, 1]
    pred_test_sorted["prob_48h"] = (1.0 - LEGACY_BLEND_48) * pred_test_sorted["prob_48h"].values + LEGACY_BLEND_48 * legacy_mean[:, 2]

pred_test_sorted["prob_72h"] = 1.0
pred_test_sorted[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = enforce_monotonicity(
    pred_test_sorted[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
)

submission = test_df[["event_id"]].merge(pred_test_sorted, on="event_id", how="left")
submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = np.clip(
    submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values,
    0.0,
    1.0,
)
submission = submission[["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]]

assert list(submission.columns) == ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert len(submission) == len(sample_sub)
assert submission["event_id"].is_unique
assert set(submission["event_id"]) == set(sample_sub["event_id"])
assert np.all((submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values >= 0.0) & (submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values <= 1.0))
assert np.all(submission["prob_12h"].values <= submission["prob_24h"].values + 1e-12)
assert np.all(submission["prob_24h"].values <= submission["prob_48h"].values + 1e-12)
assert np.all(submission["prob_48h"].values <= submission["prob_72h"].values + 1e-12)

submission.to_csv(OUTPUT_PATH, index=False)

prob72_oof = np.ones(len(train_df), dtype=float)
oof_stack = enforce_monotonicity(np.column_stack([prob12_oof, prob24_oof, prob48_oof, prob72_oof]))
local_score = compute_hybrid(oof_stack[:, 0], oof_stack[:, 1], oof_stack[:, 2], oof_stack[:, 3])

print(json.dumps({
    "data_dir": DATA_DIR,
    "legacy_files_used": len(legacy_frames),
    "local_hybrid": round(float(local_score[0]), 6),
    "local_c_index_prob12": round(float(local_score[1]), 6),
    "local_weighted_brier": round(float(local_score[2]), 6),
    "brier24": round(float(local_score[3]), 6),
    "brier48": round(float(local_score[4]), 6),
    "brier72": round(float(local_score[5]), 6),
    "submission_path": OUTPUT_PATH,
}, indent=2))

print(submission.head())


{
  "data_dir": "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
  "legacy_files_used": 0,
  "local_hybrid": 0.966821,
  "local_c_index_prob12": 0.935243,
  "local_weighted_brier": 0.019646,
  "brier24": 0.03304,
  "brier48": 0.024334,
  "brier72": 0.0,
  "submission_path": "/kaggle/working/submission.csv"
}
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.020468  0.026434  0.034498       1.0
1  13353600  0.574475  0.862602  0.905958       1.0
2  13942327  0.020587  0.027901  0.038021       1.0
3  16112781  0.678607  0.878754  0.914751       1.0
4  17132808  0.048015  0.048015  0.059718       1.0
